# 13 — Skill Gap Engine

**Day 3, Step 13.** The core logic is the plain set subtraction the Build Notes
specify — *required minus held* — applied twice, because finding F2 established
there are two kinds of skill here.

In [1]:
import sys, warnings
sys.path.insert(0, "../src"); sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# The lab imports from the factory. Nothing below reimplements pipeline logic.
from hrai.utils.config import get, raw_path, seed
from hrai.utils.io import load_raw, load_processed
from hrai.utils.logger import setup_logging
setup_logging(fmt="human")
print(f"seed={seed()}  |  datasets: {sorted(get('datasets'))}")

seed=42  |  datasets: ['employee_attrition', 'essential_skills', 'hr_performance_engagement', 'occupation_data', 'software_skills']


In [2]:
# The Build Notes' worked example, verbatim.
required = {"Python", "SQL", "MLOps", "Docker", "AWS"}
has      = {"Python", "SQL", "AWS"}
print(required - has)

{'MLOps', 'Docker'}


In [3]:
from hrai.skills.employee_skills import build_employee_skills
from hrai.skills.gap import compute_skill_gaps, per_employee_gaps

gaps = compute_skill_gaps(build_employee_skills())
f = gaps[gaps.tier == "foundational"]; t = gaps[gaps.tier == "technical"]
print(f"foundational gap rate: {f.has_gap.mean():.1%}   (graded comparison)")
print(f"technical    gap rate: {t.has_gap.mean():.1%}   (set subtraction)")

2026-08-28 01:55:10 | INFO  | crosswalk resolved


2026-08-28 01:55:10 | INFO  | role requirements built


2026-08-28 01:55:11 | INFO  | employee skills derived


2026-08-28 01:55:11 | INFO  | skill gaps computed


foundational gap rate: 9.5%   (graded comparison)
technical    gap rate: 60.0%   (set subtraction)


**Tier 2 is binary** — you either work with Kubernetes or you do not, so
this is genuine set subtraction, weighted by Hot Technology / In Demand.

**Tier 1 is graded** — nobody has *zero* Critical Thinking, so the gap is
`required_level - proficiency_level`, weighted by importance. Treating them
alike would either lose the grading or invent one.

In [4]:
per_person = per_employee_gaps(gaps)
per_person.head(10)[["person_key", "role", "gap_count", "technical_gap_count",
                     "foundational_gap_count", "gap_severity_total", "primary_gap"]]

,person_key,role,gap_count,technical_gap_count,foundational_gap_count,gap_severity_total,primary_gap
0,B-1007,Software Engineer,27,21,6,24.244,Amazon Web Services AWS software
1,B-2406,Network Engineer,27,24,3,23.778,Amazon Web Services AWS software
2,B-3498,Area Sales Manager,30,25,5,23.064,Active Listening
3,B-1054,Software Engineer,25,21,4,23.059,Atlassian JIRA
4,B-2530,Software Engineer,25,21,4,22.87,Amazon Web Services AWS software
5,B-2306,Network Engineer,27,21,6,22.748,Reading Comprehension
6,B-3996,Database Administrator,25,20,5,22.494,Amazon Web Services AWS software
7,B-1003,Software Engineer,25,21,4,22.461,Amazon Web Services AWS software
8,B-3994,Data Analyst,27,22,5,22.435,Amazon Web Services AWS software
9,B-1082,Software Engineer,25,21,4,22.414,Amazon Web Services AWS software
